# RVC — Infer giọng A → giọng model (không cần WebUI)

Dự án: Retrieval-based Voice Conversion (RVC). Notebook này tái hiện **đúng luồng** của tab *单次推理* trong `infer-web.py`: đọc audio nguồn → Hubert → (tuỳ chọn) FAISS index → đồng bộ F0 → mạng `net_g` → xuất WAV.

**Cách chạy:** mở notebook trong thư mục `rvc_standalone` (hoặc chỉnh `NOTEBOOK_ROOT` ở bước 1). Cần Python env đã cài PyTorch, fairseq, FFmpeg (như khi chạy WebUI).

**Tài nguyên:** `assets/hubert/hubert_base.pt`, `assets/rmvpe/rmvpe.pt` (nếu dùng `f0_method="rmvpe"`), file model `*.pth` trong `assets/weights`, file index `*.index` (tuỳ chọn).

## Bước 1 — Gốc thư mục & Python path

Code dưới **đưa `cwd` về `rvc_standalone`** và thêm vào `sys.path` để import được `infer`, `configs` giống `infer-web.py`. File `.env` sẽ gán `weight_root`, `index_root`, `rmvpe_root` nếu có.

In [1]:
import os
import sys

# Đường dẫn thư mục chứa notebook = gốc dự án standalone (nơi có infer/, configs/, assets/)
NOTEBOOK_ROOT = os.path.abspath(os.getcwd())
# Nếu notebook nằm nơi khác, sửa thành:
# NOTEBOOK_ROOT = r"D:\...\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone"

os.chdir(NOTEBOOK_ROOT)
if NOTEBOOK_ROOT not in sys.path:
    sys.path.insert(0, NOTEBOOK_ROOT)

from dotenv import load_dotenv

load_dotenv(os.path.join(NOTEBOOK_ROOT, ".env"), override=False)

# Fallback nếu không có .env (đường dẫn tương đối so với NOTEBOOK_ROOT)
os.environ.setdefault("weight_root", "assets/weights")
os.environ.setdefault("index_root", "logs")
os.environ.setdefault("outside_index_root", "assets/indices")
os.environ.setdefault("rmvpe_root", "assets/rmvpe")

print("cwd:", os.getcwd())
print("weight_root:", os.environ.get("weight_root"))
print("index_root:", os.environ.get("index_root"))
print("rmvpe_root:", os.environ.get("rmvpe_root"))

cwd: d:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone
weight_root: assets/weights
index_root: logs
rmvpe_root: assets/rmvpe


## Bước 2 — Tham số do bạn chỉnh (INPUT / MODEL / INDEX / Infer)

Ý nghĩa từng biến xem thêm **mục “Bảng tham số chi tiết”** ở cuối notebook.

In [2]:
# ----- Đường dẫn file -----
# Audio nguồn (giọng A): cần tồn tại trên đĩa; ffmpeg sẽ đọc và resample 16 kHz mono trong load_audio
INPUT_WAV = r"C:\Users\TEMP.THANHNGAN13.005\Downloads\16.wav"  # đổi thành đường dẫn đầy đủ của bạn

# Tên file model trong thư mục weight_root (cùng cách chọn trong dropdown WebUI)
MODEL_FILE = "giong_A_infer.pth"

# File FAISS index (.index). Để "" để tắt retrieval (chính suy luận không dùng kho embedding)
INDEX_PATH = r".\assets\indices\giong_A_added_IVF326_Flat_nprobe_1_giong_A_v2.index"

# ----- Tham số giống WebUI (单次推理) -----
SPEAKER_ID = 0       # int: id speaker trong model đa giọng (thường 0 nếu chỉ 1 speaker)
F0_UP_KEY = 0        # int: dịch pitch theo nửa cung; +12 = lên 1 quãng tám
F0_METHOD = "rmvpe"  # "pm" | "harvest" | "crepe" | "rmvpe" (GPU, cần rmvpe.pt)
INDEX_RATE = 0.75    # float 0..1: phần blending feature retrieval (0 = tắt dù vẫn có file index)
FILTER_RADIUS = 3    # int: với harvest, median filter (0–7)
RESAMPLE_SR = 0     # int Hz: 0 = giữ sample rate của model; >0 = resample đầu ra
RMS_MIX_RATE = 0.25 # float 0..1: trộn envelope RMS input vào output (1 = gần output thuần)
PROTECT = 0.33      # float 0..0.5: bảo vệ âm vô thanh; WebUI mặc định ~0.33; 0.5 = tắt cơ chế này

# Đường file F0 ngoài (tuỳ chọn). None = không gửi file, dùng F0 tự trích
F0_FILE = None

# Đường WAV xuất
OUTPUT_WAV = r".\infer_output.wav"

## Bước 3 — `Config`: thiết bị, FP16, cửa sổ infer

`Config()` đọc GPU, quyết định `device`, `is_half`, và các tham số `x_pad`, `x_query`, `x_center`, `x_max` (kích thước chunk / pad trong pipeline để vừa VRAM).

**Lưu ý Jupyter:** `Config` dùng `argparse`; kernel đôi khi truyền tham số lạ vào `sys.argv`, nên ta tạm reset `sys.argv` trước khi khởi tạo.

In [3]:
import torch

_saved_argv = sys.argv
sys.argv = ["rvc_infer_notebook"]  # tránh parse lỗi trong Jupyter
try:
    from configs.config import Config

    config = Config()
finally:
    sys.argv = _saved_argv

print("device:", config.device)
print("is_half:", config.is_half)
print("x_pad, x_query, x_center, x_max:", config.x_pad, config.x_query, config.x_center, config.x_max)

device: cuda:0
is_half: True
x_pad, x_query, x_center, x_max: 1 5 30 32


## Bước 4 — Khởi tạo `VC` và nạp trọng số model (`get_vc`)

Tương đương chọn model trong WebUI: đọc checkpoint `.pth`, dựng `net_g` (v1/v2, có/không F0), nạp `state_dict`, tạo `Pipeline` với `tgt_sr` của model.

In [22]:
from infer.modules.vc.modules import VC

vc = VC(config)

# get_vc(model_filename) — trả dict cập nhật UI trong WebUI; notebook bỏ qua giá trị trả về
_ = vc.get_vc(MODEL_FILE)

assert vc.net_g is not None, "Không load được model; kiểm tra MODEL_FILE và weight_root"
print("tgt_sr:", vc.tgt_sr)
print("version:", vc.version, "if_f0:", vc.if_f0)
print("n_spk (speaker count):", vc.cpt["config"][-3])

2026-05-03 20:51:32 | INFO | infer.modules.vc.modules | Get sid: giong_A_infer.pth
2026-05-03 20:51:32 | INFO | infer.modules.vc.modules | Loading: assets/weights/giong_A_infer.pth


2026-05-03 20:51:32 | INFO | infer.modules.vc.modules | Select index: 


tgt_sr: 40000
version: v2 if_f0: 1
n_spk (speaker count): 109


## Bước 5 — Hubert

`load_hubert` được gọi **tự động** bên trong `vc_single` nếu `hubert_model` là `None`. Ở đây có thể nạp sớm để thấy rõ một bước (tuỳ chọn).

File: `assets/hubert/hubert_base.pt`. Import `infer.modules.vc.utils` đã áp **fairseq / torch.load** tương thích PyTorch 2.6+.

In [23]:
from infer.modules.vc.utils import load_hubert

vc.hubert_model = load_hubert(config)
print("Hubert device:", next(vc.hubert_model.parameters()).device)

2026-05-03 20:51:33 | INFO | fairseq.tasks.hubert_pretraining | current directory is d:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone
2026-05-03 20:51:33 | INFO | fairseq.tasks.hubert_pretraining | HubertPretrainingTask Config {'_name': 'hubert_pretraining', 'data': 'metadata', 'fine_tuning': False, 'labels': ['km'], 'label_dir': 'label', 'label_rate': 50.0, 'sample_rate': 16000, 'normalize': False, 'enable_padding': False, 'max_keep_size': None, 'max_sample_size': 250000, 'min_sample_size': 32000, 'single_target': False, 'random_crop': True, 'pad_audio': False}
2026-05-03 20:51:33 | INFO | fairseq.models.hubert.hubert | HubertModel Config: {'_name': 'hubert', 'label_rate': 50.0, 'extractor_mode': default, 'encoder_layers': 12, 'encoder_embed_dim': 768, 'encoder_ffn_embed_dim': 3072, 'encoder_attention_heads': 12, 'activation_fn': gelu, 'layer_type': transformer, 'dropout': 0.1, 'attention_dropout': 0.1, 'activation_

Hubert device: cuda:0


## Bước 6 — Chạy infer (`vc_single`) và ghi WAV

`vc_single` nhận đúng thứ tự tham số như nút *转换* WebUI:

`sid` (SPEAKER_ID), `input_audio_path`, `f0_up_key`, `f0_file`, `f0_method`, `file_index`, `file_index2`, `index_rate`, `filter_radius`, `resample_sr`, `rms_mix_rate`, `protect`.

- Nếu bạn truyền `INDEX_PATH` vào `file_index` và đặt `file_index2` rỗng, index đó được dùng.  
- **Lưu ý:** Biểu thức `if self.tgt_sr != resample_sr >= 16000` trong code gốc là “phép toán Python” (so sánh chuỗi), không phải lỗi gõ: nhịp chuẩn suy luận là `target_sr = sample_rate của model` hoặc `resample_sr` nếu logic boolean kết quả đúng. Notebook giữ nguyên hành vi giống WebUI qua `vc_single`.

In [24]:
import soundfile as sf

file_index = INDEX_PATH.strip() if INDEX_PATH else ""
file_index2 = ""  # dùng dropdown thứ hai trong WebUI; notebook chỉ cần một ô path

info, audio_out = vc.vc_single(
    SPEAKER_ID,
    INPUT_WAV,
    F0_UP_KEY,
    F0_FILE,
    F0_METHOD,
    file_index,
    file_index2,
    INDEX_RATE,
    FILTER_RADIUS,
    RESAMPLE_SR,
    RMS_MIX_RATE,
    PROTECT,
)

print(info)

if audio_out is None:
    raise RuntimeError("Infer thất bại — xem traceback trong info ở trên.")

tgt_sr, audio_i16 = audio_out
sf.write(OUTPUT_WAV, audio_i16, tgt_sr)
print("Đã ghi:", OUTPUT_WAV, "| sr =", tgt_sr, "| mẫu:", audio_i16.shape)

2026-05-03 20:51:35 | INFO | infer.modules.vc.pipeline | Loading rmvpe model,assets/rmvpe/rmvpe.pt


Success.
Index:
.\assets\indices\giong_A_added_IVF326_Flat_nprobe_1_giong_A_v2.index.
Time:
npy: 0.15s, f0: 1.42s, infer: 0.68s.
Đã ghi: .\infer_output.wav | sr = 40000 | mẫu: (678400,)


---

## Bảng tham số chi tiết (thuyết minh)

| Tham số | Kiểu | Ý nghĩa |
|--------|------|--------|
| `NOTEBOOK_ROOT` / `cwd` | str | Mọi đường dẫn tương đối trong code RVC (hubert, rmvpe, `.env`) đều so với thư mục này. |
| `INPUT_WAV` | str | File audio nguồn. `load_audio` gọi FFmpeg, xuất **mono 16 kHz float** trước khi vào pipeline. |
| `MODEL_FILE` | str | Tên file trong `weight_root`; checkpoint chứa `config`, `weight`, `version`, `f0`. |
| `INDEX_PATH` | str | File FAISS (đuôi `.index`) huấn luyện từ embedding của **giọng đích**; dùng cho retrieval. Để rỗng: không đọc index. |
| `SPEAKER_ID` | int | Chỉ số speaker trong `emb_g` (model đa người nói). Single-speaker thường là `0`. |
| `F0_UP_KEY` | int | Dịch tông theo **nửa cung** (semitone): `12` tương đương 1 quãng tám. |
| `F0_METHOD` | str | Cách ước lượng pitch: `pm` (nhanh), `harvest` (thấp tốt, chậm), `crepe` (nặng GPU), `rmvpe` (RVC mặc định hay dùng, cần `rmvpe.pt`). |
| `INDEX_RATE` | float ∈ [0,1] | Trong vòng lặp retrieval: `feats = index_rate * feats_retrieved + (1-index_rate) * feats_hubert`. `0` tắt trộn dù file index có tồn tại (điều kiện trong pipeline: cần `index_rate != 0` và file tồn tại). |
| `FILTER_RADIUS` | int | Với `harvest`: median filter bán kính (giảm “im lặng” giả). |
| `RESAMPLE_SR` | int | `0`: giữ `tgt_sr` của model; nếu biểu thức legacy trong `vc_single` chọn `resample_sr`, đầu ra có thể resample thêm (hành vi giống WebUI). |
| `RMS_MIX_RATE` | float | Trộn **bao envelope RMS** của tín hiệu vào và ra (tự nhiên hơn khi < 1). |
| `PROTECT` | float ∈ [0, 0.5] | Bảo vệ vùng F0 thấp / âm vô thanh: kết hợp `feats` gốc và đã chỉnh pitch; `0.5` trong WebUI nghĩa tắt mạnh phần “bảo vệ” theo ghi chú UI. |
| `F0_FILE` | path hoặc None | File cung cấp đường cong F0 thay cho ước lượng (định dạng theo code pipeline: từng dòng có thể là cặp thời gian, pitch). |
| `OUTPUT_WAV` | str | File WAV ở sample rate `tgt_sr` (sau xử lý), dạng **int16** như internal pipeline. |

### Các thành phần mạng (không chỉnh từ notebook trừ khi đổi code)

- **Hubert:** trích đặc trưng content từ audio 16 kHz; lớp `output_layer` phụ thuộc `version` (v1: 9, v2: 12).  
- **Retrieval (FAISS):** so khớp embedding hiện tại với kho embedding đã train; trung bình có trọng số theo kNN (trong code: `k=8`).  
- **net_g:** vocoder/acoustic model có điều kiện pitch (nếu `if_f0==1`) và speaker ID.

### Khắc phục sự cố nhanh

- `Weights only load failed` / fairseq: đảm bảo dùng `infer` trong `rvc_standalone` (có `fairseq_torch_load_compat`).  
- `path does not exist`: kiểm tra `INPUT_WAV`.  
- Lỗi RMVPE / CUDA: thử `F0_METHOD = "pm"` hoặc giảm độ dài file input.